# Absorption Spectrum of a Molecular Dimer

This notebook demonstrates the full workflow for computing and visualising the linear absorption spectrum of a coupled molecular dimer using **quantarhei**.

## Workflow
1. Create two molecules (homodimer, 12000 cm⁻¹ transition energy)
2. Assemble them into an aggregate with resonance coupling J = 100 cm⁻¹
3. Define a bath via an overdamped Brownian oscillator correlation function
4. Calculate the absorption spectrum
5. Plot and compare with/without coupling to reveal the **Davydov splitting**

The key physical result: turning on the resonance coupling splits the single absorption peak into two exciton peaks separated by approximately 2J.

In [1]:
%config InlineBackend.figure_format = 'svg'
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for reproducibility
import matplotlib.pyplot as plt
import numpy as np
import quantarhei as qr

## 1. Create the Molecules

Each molecule is a two-level system. Both have the same electronic transition at 12000 cm⁻¹ (a homodimer).
Transition dipole moments are set perpendicular to each other, giving equal oscillator strength to both exciton states.

In [2]:
# Both molecules have a ground state (0) and excited state (1) at 12000 cm^-1
with qr.energy_units("1/cm"):
    m1 = qr.Molecule(name="M1", elenergies=[0.0, 12000.0])
    m2 = qr.Molecule(name="M2", elenergies=[0.0, 12000.0])

# Orthogonal transition dipoles: both exciton states will be optically active
m1.set_dipole(0, 1, [1.0, 0.0, 0.0])
m2.set_dipole(0, 1, [0.0, 1.0, 0.0])

print("M1 transition energy: 12000 cm^-1, dipole along x")
print("M2 transition energy: 12000 cm^-1, dipole along y")

M1 transition energy: 12000 cm^-1, dipole along x
M2 transition energy: 12000 cm^-1, dipole along y


## 2. Build the Aggregate

The aggregate Hamiltonian includes a resonance coupling J = 100 cm⁻¹ between the two molecules.
This off-diagonal coupling mixes the site states into delocalised **exciton states** split by 2J ≈ 200 cm⁻¹.

In [3]:
agg = qr.Aggregate(name="Homodimer")
agg.add_Molecule(m1)
agg.add_Molecule(m2)

# Set resonance coupling between molecule 0 and molecule 1
with qr.energy_units("1/cm"):
    agg.set_resonance_coupling(0, 1, 100.0)

print("Aggregate assembled with J = 100 cm^-1 resonance coupling")

Aggregate assembled with J = 100 cm^-1 resonance coupling


## 3. Attach the Bath (System–Bath Coupling)

Each molecule is coupled to an independent harmonic bath described by an **overdamped Brownian oscillator** correlation function:

$$C(t) = \frac{2\lambda k_B T}{\hbar \tau_c} e^{-t/\tau_c} - i \frac{\lambda}{\tau_c} e^{-t/\tau_c}$$

Parameters:
- Reorganisation energy λ = 20 cm⁻¹
- Correlation time τ_c = 100 fs
- Temperature T = 300 K
- 20 Matsubara terms for thermal convergence

In [4]:
# Time axis: 1000 steps of 1 fs
ta = qr.TimeAxis(0.0, 1000, 1.0)

params = {
    "ftype": "OverdampedBrownian",
    "reorg": 20.0,    # reorganization energy in 1/cm
    "cortime": 100.0,  # correlation time in fs
    "T": 300.0,        # temperature in Kelvin
    "matsubara": 20    # number of Matsubara terms
}

with qr.energy_units("1/cm"):
    cf = qr.CorrelationFunction(ta, params)

# Attach the same bath to both molecules (independent identical baths)
m1.set_transition_environment((0, 1), cf)
m2.set_transition_environment((0, 1), cf)

print("Bath attached: overdamped Brownian, lambda=20 cm^-1, tau_c=100 fs, T=300 K")

Bath attached: overdamped Brownian, lambda=20 cm^-1, tau_c=100 fs, T=300 K


## 4. Build and Calculate the Absorption Spectrum

`agg.build(mult=1)` constructs the Frenkel exciton Hamiltonian in the single-excitation manifold.
`AbsSpectrumCalculator` computes the linear response function and Fourier-transforms it to the frequency domain.
The `rwa` (rotating wave approximation) frequency centres the calculation around the transition frequency.

In [5]:
# Build the Hamiltonian in the single-excitation manifold
agg.build(mult=1)

# Set up the absorption spectrum calculator
ac = qr.AbsSpectrumCalculator(ta, system=agg)

# Bootstrap with RWA frequency at the monomer transition (12000 cm^-1 -> internal units)
with qr.energy_units("1/cm"):
    ac.bootstrap(rwa=12000.0)
    a = ac.calculate()

# Convert frequency axis from internal units to cm^-1
freq = qr.convert(a.axis.data, "int", "1/cm")

print(f"Spectrum computed: {len(freq)} points, "
      f"range {freq.min():.0f}\u2013{freq.max():.0f} cm^-1")

Spectrum computed: 1000 points, range 3661–20322 cm^-1


## 5. Plot the Absorption Spectrum

We zoom into the region around the transition (11500–12500 cm⁻¹) to clearly see the exciton peaks.

In [6]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(freq, a.data / np.max(a.data), color="steelblue", lw=2)
ax.set_xlabel("Frequency (1/cm)")
ax.set_ylabel("Normalized Intensity")
ax.set_title(r"Absorption Spectrum of Homodimer ($J = 100\,\mathrm{cm}^{-1}$)")
ax.set_xlim(11500, 12500)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

/var/folders/vx/gjt_y6_50z3b41b0w7yk4ryc0000gn/T/ipykernel_61778/1865955747.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Davydov Splitting: Comparison With and Without Coupling

To see the Davydov splitting explicitly, we build a second aggregate with the coupling set to zero and compare both spectra side by side.

With J = 0 all oscillator strength sits at 12000 cm⁻¹. With J = 100 cm⁻¹ the exciton states split to roughly 12000 ± 100 cm⁻¹, creating two peaks separated by ∼200 cm⁻¹ (= 2J).

In [7]:
# --- Build dimer WITHOUT coupling (J = 0) ---
with qr.energy_units("1/cm"):
    m1_nc = qr.Molecule(name="M1", elenergies=[0.0, 12000.0])
    m2_nc = qr.Molecule(name="M2", elenergies=[0.0, 12000.0])
    cf_nc = qr.CorrelationFunction(ta, params)

m1_nc.set_dipole(0, 1, [1.0, 0.0, 0.0])
m2_nc.set_dipole(0, 1, [0.0, 1.0, 0.0])
m1_nc.set_transition_environment((0, 1), cf_nc)
m2_nc.set_transition_environment((0, 1), cf_nc)

agg_nc = qr.Aggregate(name="NoCoupling")
agg_nc.add_Molecule(m1_nc)
agg_nc.add_Molecule(m2_nc)
agg_nc.build(mult=1)

ac_nc = qr.AbsSpectrumCalculator(ta, system=agg_nc)
with qr.energy_units("1/cm"):
    ac_nc.bootstrap(rwa=12000.0)
    a_nc = ac_nc.calculate()

freq_nc = qr.convert(a_nc.axis.data, "int", "1/cm")

# --- Side-by-side comparison ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(freq_nc, a_nc.data / np.max(a_nc.data),
             color="gray", lw=2)
axes[0].set_xlabel("Frequency (1/cm)")
axes[0].set_ylabel("Normalized Intensity")
axes[0].set_title("No Coupling (J = 0)")
axes[0].set_xlim(11500, 12500)
axes[0].grid(True, alpha=0.3)

axes[1].plot(freq, a.data / np.max(a.data),
             color="steelblue", lw=2)
axes[1].set_xlabel("Frequency (1/cm)")
axes[1].set_ylabel("Normalized Intensity")
axes[1].set_title(r"With Coupling ($J = 100\,\mathrm{cm}^{-1}$) \u2014 Davydov Splitting")
axes[1].set_xlim(11500, 12500)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

/var/folders/vx/gjt_y6_50z3b41b0w7yk4ryc0000gn/T/ipykernel_61778/3381531291.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

| Parameter | Value |
|-----------|-------|
| Transition energy | 12000 cm⁻¹ |
| Resonance coupling J | 100 cm⁻¹ |
| Bath type | Overdamped Brownian oscillator |
| Reorganisation energy λ | 20 cm⁻¹ |
| Correlation time τ_c | 100 fs |
| Temperature | 300 K |

The right panel shows the **Davydov splitting**: the resonance coupling delocalises the excitation over both molecules, creating two exciton eigenstates at energies E± ≈ ε ± J.
Because the dipoles are orthogonal (equal projections onto the exciton eigenvectors), both peaks carry equal oscillator strength.
The lineshape broadening is controlled by the bath reorganisation energy and correlation time.